# Jaden Konekté — entraînement du CNN sur GPU (Colab)

Même entraînement que `ia/cnn/entrainement.py` en local, mais sur GPU : environ 20 minutes au lieu de plusieurs heures.

1. **Exécution → Modifier le type d'exécution → GPU (T4)**.
2. **Exécution → Tout exécuter**.
3. À la fin, le navigateur télécharge `modele_cnn.zip` : `tomate_mobilenetv3.onnx`, `tomate_mobilenetv3.onnx.data`,
   `classes.json` et `evaluation.md`.

Données : PlantVillage (GitHub spMohanty), PlantDoc (CC BY 4.0, GitHub pratikkayal), deux jeux Mendeley Data
(CC BY 4.0) : Pakistan (DOI 10.17632/3mbnb82mxd.2) et Bangladesh (DOI 10.17632/93h9p62kg4.1).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "PAS DE GPU : changer le type d'exécution"
!pip -q install onnx onnxruntime onnxscript
!mkdir -p /content/jk/ia/cnn /content/jk/service-ia/ia_service /content/externe

## Scripts du projet

In [ ]:
%%writefile /content/jk/ia/cnn/preparer_donnees.py
"""
Prépare le sous-ensemble « tomate » de PlantVillage en dossiers train/ et val/.

1. Téléchargez PlantVillage (version « color »), par exemple depuis Kaggle
   (jeu « PlantVillage Dataset ») ou via TensorFlow Datasets (plant_village).
2. Lancez :
       python preparer_donnees.py --source chemin/vers/plantvillage/color --sortie donnees
   Seuls les dossiers dont le nom commence par « Tomato » sont conservés.
"""
import argparse
import random
import shutil
from pathlib import Path


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--source", required=True)
    ap.add_argument("--sortie", default="donnees")
    ap.add_argument("--val", type=float, default=0.2, help="part des images en validation")
    ap.add_argument("--prefixe", default="Tomato")
    a = ap.parse_args()
    random.seed(0)

    dossiers = sorted(d for d in Path(a.source).iterdir() if d.is_dir() and d.name.startswith(a.prefixe))
    if not dossiers:
        raise SystemExit(f"Aucun dossier « {a.prefixe}* » dans {a.source}")
    for d in dossiers:
        images = sorted(p for p in d.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"})
        random.shuffle(images)
        n_val = int(len(images) * a.val)
        for i, img in enumerate(images):
            dest = Path(a.sortie) / ("val" if i < n_val else "train") / d.name
            dest.mkdir(parents=True, exist_ok=True)
            shutil.copy2(img, dest / img.name)
        print(f"{d.name:<55} {len(images) - n_val:>5} train  {n_val:>5} val")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/jk/ia/cnn/preparer_plantdoc.py
"""Range les photos de tomate PlantDoc (prises au champ) sous les noms de classes PlantVillage.

PlantDoc (Singh et al., 2020, licence CC BY 4.0) : https://github.com/pratikkayal/PlantDoc-Dataset
Ses photos viennent d'internet : fonds chargés, plusieurs feuilles, éclairages variés. Elles servent :
  * à l'entraînement (partie train), pour que le modèle ne dépende plus du fond uni de PlantVillage ;
  * à l'évaluation sur photos réelles (partie test), jamais vue à l'entraînement.

    python preparer_plantdoc.py --source <dossier PlantDoc extrait> --sortie donnees_terrain

PlantDoc n'a pas de classe « tache cible » (Target_Spot) : cette classe reste apprise sur PlantVillage seul.
"""
import argparse
import hashlib
import random
import shutil
from pathlib import Path

from PIL import Image

CORRESPONDANCE = {
    "Tomato leaf bacterial spot": "Tomato___Bacterial_spot",
    "Tomato Early blight leaf": "Tomato___Early_blight",
    "Tomato leaf late blight": "Tomato___Late_blight",
    "Tomato mold leaf": "Tomato___Leaf_Mold",
    "Tomato Septoria leaf spot": "Tomato___Septoria_leaf_spot",
    "Tomato two spotted spider mites leaf": "Tomato___Spider_mites Two-spotted_spider_mite",
    "Tomato leaf yellow virus": "Tomato___Tomato_Yellow_Leaf_Curl_Virus",
    "Tomato leaf mosaic virus": "Tomato___Tomato_mosaic_virus",
    "Tomato leaf": "Tomato___healthy",
}


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--source", required=True, help="dossier contenant train/ et test/ de PlantDoc")
    ap.add_argument("--sortie", default="donnees_terrain")
    ap.add_argument("--part-test", type=float, default=0.3,
                    help="part des photos « train » de PlantDoc ajoutées au test (le test officiel n'a que 69 photos)")
    a = ap.parse_args()
    random.seed(0)
    vues, comptes, illisibles = set(), {"train": 0, "test": 0}, 0
    for partie in ("test", "train"):          # test d'abord : un doublon présent dans les deux parties reste en test
        for dossier, classe in CORRESPONDANCE.items():
            src = Path(a.source) / partie / dossier
            if not src.is_dir():
                continue
            fichiers = []
            for f in sorted(src.iterdir()):
                empreinte = hashlib.sha1(f.read_bytes()).hexdigest()
                if empreinte in vues:             # même image publiée deux fois : on ne la garde qu'une fois
                    continue
                vues.add(empreinte)
                try:
                    with Image.open(f) as img:
                        img.convert("RGB")
                except Exception:                 # fichier qui n'est pas une image lisible
                    illisibles += 1
                    continue
                fichiers.append(f)
            random.shuffle(fichiers)
            nb_test = len(fichiers) if partie == "test" else round(len(fichiers) * a.part_test)
            for i, f in enumerate(fichiers):
                cible = "test" if i < nb_test else "train"
                dst = Path(a.sortie) / cible / classe
                dst.mkdir(parents=True, exist_ok=True)
                shutil.copy2(f, dst / f.name)
                comptes[cible] += 1
    print(f"entraînement : {comptes['train']} photos · test : {comptes['test']} photos · illisibles écartées : {illisibles}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/jk/ia/cnn/preparer_mendeley.py
"""Range deux jeux de photos prises au champ (Mendeley Data, licence CC BY 4.0) sous les noms de classes PlantVillage.

  * Malik, Ali, Ali, « Tomato Leaf Disease Classification Dataset in Pakistan », v2, DOI 10.17632/3mbnb82mxd.2
    Seul le dossier « Dataset (raw) » est utilisé (830 photos) : le dossier « Augmented Dataset » ne contient que
    des copies retouchées de ces mêmes photos, qui fausseraient la séparation entraînement / test.
  * « Tomato leaf diseases » (Bangladesh), v1, DOI 10.17632/93h9p62kg4.1
    Classes gardées : TYLCV, acariens, cladosporiose, mildiou, alternariose, gale bactérienne, saine.
    Écartées faute d'équivalent : mineuse, dégâts d'insectes, cercosporiose, « autres ».

    python preparer_mendeley.py --pakistan "<...>/Tomato Dataset/Dataset (raw)" --bangladesh <dossier> --sortie donnees_mendeley

Une part des photos (20 % par défaut) est mise de côté pour l'évaluation (evaluer.py) et jamais vue à l'entraînement.
Les doublons exacts (même fichier) sont écartés, y compris ceux déjà présents dans PlantDoc.
"""
import argparse
import hashlib
import os
import random
from pathlib import Path

from PIL import Image, ImageOps

PAKISTAN = {
    "Tomato_Early_blight_leaf": "Tomato___Early_blight",
    "Tomato_Healthy_leaf": "Tomato___healthy",
    "Tomato_Septoria_leaf_spot": "Tomato___Septoria_leaf_spot",
    "Tomato_leaf_late_blight": "Tomato___Late_blight",
    "Tomato_leaf_yellow_curl_virus": "Tomato___Tomato_Yellow_Leaf_Curl_Virus",
    "Tomato_mold_leaf": "Tomato___Leaf_Mold",
}
BANGLADESH = {
    "Bacterial_Spot": "Tomato___Bacterial_spot",
    "Early_Blight": "Tomato___Early_blight",
    "Healthy": "Tomato___healthy",
    "Late_Blight": "Tomato___Late_blight",
    "Leaf_Mold": "Tomato___Leaf_Mold",
    "Spider_Mites": "Tomato___Spider_mites Two-spotted_spider_mite",
    "Tomato_Leaf_Curl_Virus": "Tomato___Tomato_Yellow_Leaf_Curl_Virus",
}
EXTENSIONS = {".jpg", ".jpeg", ".png"}
COTE_MAX = 640       # les photos de téléphone (12 Mpx) sont réduites : le modèle travaille en 224 px


def chemin_long(p: str) -> Path:
    """Sous Windows, certains noms de fichiers du jeu Pakistan dépassent 260 caractères : chemin étendu."""
    absolu = os.path.abspath(p)
    return Path("\\\\?\\" + absolu if os.name == "nt" and not absolu.startswith("\\\\") else absolu)


def empreinte(f: Path) -> str:
    return hashlib.sha1(f.read_bytes()).hexdigest()


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--pakistan", required=True, help="dossier « Dataset (raw) » du jeu Pakistan")
    ap.add_argument("--bangladesh", required=True, help="dossier du jeu Bangladesh (un sous-dossier par classe)")
    ap.add_argument("--sortie", default="donnees_mendeley")
    ap.add_argument("--deja-vus", default="donnees_terrain", help="photos déjà utilisées (PlantDoc), à ne pas recopier")
    ap.add_argument("--part-test", type=float, default=0.2)
    a = ap.parse_args()
    random.seed(0)
    vues = {empreinte(f) for f in Path(a.deja_vus).rglob("*") if f.is_file()} if Path(a.deja_vus).is_dir() else set()
    comptes, ecartees = {"train": 0, "test": 0}, 0
    for prefixe, racine, correspondance in (("pk", a.pakistan, PAKISTAN), ("bd", a.bangladesh, BANGLADESH)):
        for dossier, classe in correspondance.items():
            src = chemin_long(racine) / dossier
            fichiers = []
            for f in sorted(src.iterdir()) if src.is_dir() else []:
                if f.suffix.lower() not in EXTENSIONS:
                    continue
                e = empreinte(f)
                if e in vues:
                    ecartees += 1
                    continue
                vues.add(e)
                try:
                    with Image.open(f) as img:
                        img.convert("RGB")
                except Exception:              # fichier illisible
                    ecartees += 1
                    continue
                fichiers.append((f, e))
            random.shuffle(fichiers)
            nb_test = round(len(fichiers) * a.part_test)
            for i, (f, e) in enumerate(fichiers):
                cible = "test" if i < nb_test else "train"
                dst = Path(a.sortie) / cible / classe
                dst.mkdir(parents=True, exist_ok=True)
                with Image.open(f) as img:
                    img = ImageOps.exif_transpose(img).convert("RGB")
                    img.thumbnail((COTE_MAX, COTE_MAX))
                    img.save(dst / f"{prefixe}_{e[:16]}.jpg", quality=92)   # nom court, sans collision
                comptes[cible] += 1
    print(f"entraînement : {comptes['train']} photos · test : {comptes['test']} photos · "
          f"doublons ou illisibles écartés : {ecartees}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/jk/ia/cnn/entrainement.py
"""
Diagnostic des maladies de la tomate — entraînement d'un CNN léger et export ONNX.

Version 2 : le premier modèle, appris sur PlantVillage seul (feuilles isolées sur fond uni, en laboratoire),
reconnaissait mal les vraies photos. Trois changements :
  1. photos réelles PlantDoc ajoutées à l'entraînement (preparer_plantdoc.py), et surreprésentées ;
  2. fond uni des images PlantVillage remplacé au hasard par un fond flou, et augmentations plus fortes
     (recadrages serrés, rotations, perspective, flou, couleurs) ;
  3. MobileNetV3-Large au lieu de Small, classes rééquilibrées (le virus TYLCV était 14 fois plus fréquent).

Données (une classe = un dossier, noms de classes PlantVillage) :
    donnees/train, donnees/val                 PlantVillage   (preparer_donnees.py)
    donnees_terrain/train, donnees_terrain/test PlantDoc       (preparer_plantdoc.py)
    donnees_mendeley/train, donnees_mendeley/test Mendeley Data (preparer_mendeley.py)

    pip install torch torchvision onnx onnxruntime onnxscript
    python entrainement.py --epoques 8 --images-par-epoque 6000 --sortie candidat/tomate_mobilenetv3.onnx

Le choix de la meilleure époque se fait sur une partie des photos PlantDoc d'entraînement mise de côté :
le jeu PlantDoc « test » reste intact pour l'évaluation finale (evaluer.py).
"""
import argparse
import json
import os
import random
import time
from collections import Counter
from pathlib import Path

import numpy as np
import torch
from PIL import Image, ImageFilter
from torch import nn
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import models, transforms

TAILLE = 224
MOY, ECART = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]  # normalisation ImageNet
EXTENSIONS = {".jpg", ".jpeg", ".png"}


def lister(dossier: Path, classes: list[str], max_par_classe: int = 0) -> list[tuple[Path, int]]:
    """(chemin, numéro de classe) pour chaque image ; une classe absente du dossier est simplement ignorée."""
    elements = []
    for i, nom in enumerate(classes):
        fichiers = sorted(f for f in (dossier / nom).glob("*") if f.suffix.lower() in EXTENSIONS) \
            if (dossier / nom).is_dir() else []
        if max_par_classe and len(fichiers) > max_par_classe:
            fichiers = random.sample(fichiers, max_par_classe)
        elements += [(f, i) for f in fichiers]
    return elements


def remplacer_fond(img: Image.Image, fonds: list[Path]) -> Image.Image:
    """Remplace le fond uni d'une image PlantVillage par un fond flou tiré d'une photo réelle, ou un dégradé.

    Les fonds PlantVillage sont gris (peu saturés) alors que la feuille est verte, jaune ou brune : la feuille
    est repérée par sa saturation, puis le masque est nettoyé (fermeture des trous, suppression des points isolés).
    Si le masque paraît faux (presque vide ou presque plein), l'image est laissée telle quelle.
    """
    saturation = np.asarray(img.convert("HSV"))[..., 1]
    masque = Image.fromarray(((saturation > 45) * 255).astype(np.uint8))
    masque = masque.filter(ImageFilter.MedianFilter(7))
    masque = masque.filter(ImageFilter.MaxFilter(9)).filter(ImageFilter.MinFilter(9))     # bouche les trous
    masque = masque.filter(ImageFilter.MinFilter(5)).filter(ImageFilter.MaxFilter(5))     # retire les points
    part = np.asarray(masque).mean() / 255
    if not 0.15 < part < 0.9:
        return img
    masque = masque.filter(ImageFilter.GaussianBlur(1.5))
    if fonds and random.random() < 0.7:
        with Image.open(random.choice(fonds)) as f:
            nouveau = f.convert("RGB").resize(img.size).filter(ImageFilter.GaussianBlur(random.uniform(4, 12)))
    else:
        c1, c2 = np.random.randint(0, 256, 3), np.random.randint(0, 256, 3)
        t = np.linspace(0, 1, img.size[1])[:, None, None]
        nouveau = Image.fromarray((c1 * (1 - t) + c2 * t).repeat(img.size[0], 1).astype(np.uint8))
    return Image.composite(img, nouveau, masque)


class Images(Dataset):
    def __init__(self, elements, transformation, fonds=None, p_fond=0.0):
        self.elements, self.tf, self.fonds, self.p_fond = elements, transformation, fonds or [], p_fond

    def __len__(self):
        return len(self.elements)

    def __getitem__(self, i):
        chemin, classe = self.elements[i]
        with Image.open(chemin) as f:
            img = f.convert("RGB")
        if self.p_fond and random.random() < self.p_fond:
            img = remplacer_fond(img, self.fonds)
        return self.tf(img), classe


def exactitude(modele, dl, dev) -> float:
    modele.eval()
    ok = n = 0
    with torch.no_grad():
        for x, y in dl:
            ok += (modele(x.to(dev)).argmax(1).cpu() == y).sum().item()
            n += len(y)
    return ok / n if n else 0.0


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--donnees", default="donnees")
    ap.add_argument("--terrain", nargs="+", default=["donnees_terrain", "donnees_mendeley"],
                    help="dossiers de photos réelles (chacun avec train/ et test/)")
    ap.add_argument("--architecture", choices=["large", "small"], default="large")
    ap.add_argument("--epoques", type=int, default=8)
    ap.add_argument("--lot", type=int, default=32)
    ap.add_argument("--images-par-epoque", type=int, default=6000)
    ap.add_argument("--max-par-classe", type=int, default=1500, help="images PlantVillage gardées par classe")
    ap.add_argument("--poids-terrain", type=float, default=5.0, help="surreprésentation des photos réelles")
    ap.add_argument("--sortie", default="candidat/tomate_mobilenetv3.onnx")
    ap.add_argument("--exporter-seulement", action="store_true",
                    help="exporte en ONNX le point de sauvegarde existant sans réentraîner")
    ap.add_argument("--processus", type=int, default=2, help="processus de chargement des images")
    a = ap.parse_args()
    random.seed(0)
    np.random.seed(0)
    torch.manual_seed(0)
    torch.set_num_threads(os.cpu_count() or 4)
    dev = "cuda" if torch.cuda.is_available() else "cpu"

    classes = sorted(d.name for d in (Path(a.donnees) / "train").iterdir() if d.is_dir())
    labo = lister(Path(a.donnees) / "train", classes, a.max_par_classe)
    terrain = [e for d in a.terrain if (Path(d) / "train").is_dir() for e in lister(Path(d) / "train", classes)]
    random.shuffle(terrain)
    nb_choix = len(terrain) // 7                       # ~15 % des photos réelles réservées au choix de l'époque
    terrain_choix, terrain = terrain[:nb_choix], terrain[nb_choix:]
    val_labo = lister(Path(a.donnees) / "val", classes, 60)
    print(f"{len(classes)} classes · entraînement : {len(labo)} images labo + {len(terrain)} photos réelles · "
          f"choix de l'époque : {len(terrain_choix)} photos réelles + {len(val_labo)} images labo", flush=True)

    tf_train = transforms.Compose([
        transforms.RandomResizedCrop(TAILLE, scale=(0.25, 1.0)),
        transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(),
        transforms.RandomApply([transforms.RandomRotation(30)], p=0.5),
        transforms.RandomPerspective(0.2, p=0.3),
        transforms.ColorJitter(0.4, 0.4, 0.4, 0.04),
        transforms.RandomApply([transforms.GaussianBlur(7, sigma=(0.1, 2.0))], p=0.3),
        transforms.ToTensor(), transforms.Normalize(MOY, ECART),
        transforms.RandomErasing(p=0.2, scale=(0.02, 0.12))])
    tf_val = transforms.Compose([
        transforms.Resize(256), transforms.CenterCrop(TAILLE),
        transforms.ToTensor(), transforms.Normalize(MOY, ECART)])

    fonds = [f for f, _ in terrain]
    ds_labo = Images(labo, tf_train, fonds, p_fond=0.6)
    ds_terrain = Images(terrain, tf_train)
    ds_train = torch.utils.data.ConcatDataset([ds_labo, ds_terrain])
    # Échantillonnage : chaque classe pèse autant ; avec poids-terrain = 5, environ 55 % des images vues
    # à chaque époque sont des photos réelles (PlantDoc et Mendeley)
    n_labo, n_terrain = Counter(c for _, c in labo), Counter(c for _, c in terrain)
    poids = [1 / n_labo[c] for _, c in labo] + [a.poids_terrain / max(n_terrain[c], 1) / 4 for _, c in terrain]
    sampler = WeightedRandomSampler(poids, a.images_par_epoque, replacement=True)
    dl_train = DataLoader(ds_train, batch_size=a.lot, sampler=sampler, num_workers=a.processus, persistent_workers=True)
    dl_choix = DataLoader(Images(terrain_choix, tf_val), batch_size=64, num_workers=1)
    dl_labo = DataLoader(Images(val_labo, tf_val), batch_size=64, num_workers=1)

    if a.architecture == "large":
        modele = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.DEFAULT)
    else:
        modele = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.DEFAULT)
    modele.classifier[3] = nn.Linear(modele.classifier[3].in_features, len(classes))
    modele.to(dev)

    opt = torch.optim.AdamW(modele.parameters(), lr=6e-4, weight_decay=1e-4)
    pas_total = a.epoques * len(dl_train)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=6e-4, total_steps=pas_total, pct_start=0.15)
    perte = nn.CrossEntropyLoss(label_smoothing=0.1)

    sortie = Path(a.sortie)
    sortie.parent.mkdir(parents=True, exist_ok=True)
    sauvegarde = sortie.with_suffix(".pt")      # meilleur état, écrit à chaque progrès : un plantage ne perd rien
    meilleur, meilleur_etat = -1.0, None
    if a.exporter_seulement:
        a.epoques, meilleur_etat = 0, torch.load(sauvegarde, map_location="cpu")
    for ep in range(1, a.epoques + 1):
        debut = time.time()
        modele.train()
        for x, y in dl_train:
            x, y = x.to(dev), y.to(dev)
            opt.zero_grad()
            perte(modele(x), y).backward()
            opt.step()
            sched.step()
        reel, labo_ok = exactitude(modele, dl_choix, dev), exactitude(modele, dl_labo, dev)
        marque = ""
        if reel > meilleur:
            meilleur, meilleur_etat, marque = reel, {k: v.detach().clone() for k, v in modele.state_dict().items()}, "  ← meilleure"
            torch.save(meilleur_etat, sauvegarde)
        print(f"époque {ep}/{a.epoques} ({time.time() - debut:.0f} s) — photos réelles : {reel:.3f} · "
              f"labo : {labo_ok:.3f}{marque}", flush=True)

    modele.load_state_dict(meilleur_etat)
    modele.eval().cpu()
    torch.onnx.export(
        modele, torch.randn(1, 3, TAILLE, TAILLE), str(sortie),
        input_names=["image"], output_names=["logits"],
        dynamic_axes={"image": {0: "lot"}, "logits": {0: "lot"}}, opset_version=17)
    import onnx                      # architecture notée dans le fichier : le service l'affiche dans la version
    proto = onnx.load(str(sortie), load_external_data=False)
    onnx.helper.set_model_props(proto, {"architecture": f"mobilenetv3-{a.architecture}"})
    onnx.save(proto, str(sortie))
    (sortie.parent / "classes.json").write_text(json.dumps(classes, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"Modèle exporté : {sortie} (architecture {a.architecture})")

    # Vérification : mêmes sorties avec ONNX Runtime
    import onnxruntime as ort
    x = torch.randn(2, 3, TAILLE, TAILLE)
    ref = modele(x).detach().numpy()
    out = ort.InferenceSession(str(sortie)).run(None, {"image": x.numpy()})[0]
    print("écart max PyTorch / ONNX :", float(np.abs(ref - out).max()))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/jk/ia/cnn/evaluer.py
"""Évaluation du modèle EXPORTÉ (ONNX) avec le prétraitement du service de diagnostic.

On mesure la chaîne réellement utilisée en production (service-ia), et non le modèle PyTorch :
une différence de prétraitement ou d'export se verrait ici. Trois jeux d'images :
  * PlantVillage (validation) : photos de laboratoire, fond uni ;
  * PlantDoc (test) : photos réelles collectées sur internet, jamais vues à l'entraînement ;
  * Mendeley (test) : photos prises au champ au Pakistan et au Bangladesh, jamais vues à l'entraînement.

    python evaluer.py --modele tomate_mobilenetv3.onnx > evaluation.md
    python evaluer.py --modele nouveau.onnx --ancien tomate_mobilenetv3.onnx   # comparaison
"""
import argparse
import json
import sys
import time
from pathlib import Path

import numpy as np

sys.path.insert(0, str(Path(__file__).resolve().parents[2] / "service-ia"))
from ia_service.classifieur import softmax  # noqa: E402
from ia_service.images import charger, pretraiter  # noqa: E402

JEUX = {"PlantVillage — validation (laboratoire)": "donnees/val",
        "PlantDoc — test (photos réelles)": "donnees_terrain/test",
        "Mendeley — test (photos réelles prises au champ)": "donnees_mendeley/test"}
SEUIL = 0.7


def evaluer(chemin_modele: str, dossier: Path, classes: list[str]) -> dict:
    """Prédit chaque image du dossier (un sous-dossier par classe) et renvoie la matrice de confusion."""
    import onnxruntime as ort
    session = ort.InferenceSession(chemin_modele, providers=["CPUExecutionProvider"])
    entree = session.get_inputs()[0].name
    n = len(classes)
    confusion = np.zeros((n, n), dtype=int)
    justes, faux, durees = [], [], []
    for vraie, nom in enumerate(classes):
        if not (dossier / nom).is_dir():
            continue
        for f in sorted((dossier / nom).iterdir()):
            try:
                x = pretraiter(charger(f.read_bytes()))
            except Exception:           # image refusée par le service (trop petite, format…) : ignorée
                continue
            debut = time.perf_counter()
            p = softmax(session.run(None, {entree: x})[0][0].astype(np.float64))
            durees.append(time.perf_counter() - debut)
            predite = int(p.argmax())
            confusion[vraie, predite] += 1
            (justes if predite == vraie else faux).append(float(p[predite]))
    return {"confusion": confusion, "justes": justes, "faux": faux, "durees": durees}


def resume(r: dict) -> dict:
    c = r["confusion"]
    total = c.sum()
    fiables = [s for s in r["justes"] + r["faux"] if s >= SEUIL]
    fiables_justes = sum(s >= SEUIL for s in r["justes"])
    return {"images": int(total), "exactitude": np.trace(c) / total if total else 0,
            "fiables": len(fiables) / total if total else 0,
            "exactitude_fiables": fiables_justes / len(fiables) if fiables else 0}


def tableau_classes(r: dict, classes, libelles) -> str:
    c = r["confusion"]
    lignes = ["| Classe | Images | Précision | Rappel | F1 |", "|---|---|---|---|---|"]
    for i, nom in enumerate(classes):
        if not c[i].sum():
            continue
        vp = c[i, i]
        precision = vp / c[:, i].sum() if c[:, i].sum() else 0
        rappel = vp / c[i].sum()
        f1 = 2 * precision * rappel / (precision + rappel) if precision + rappel else 0
        lignes.append(f"| {libelles.get(nom, nom)} | {c[i].sum()} | {precision:.1%} | {rappel:.1%} | {f1:.2f} |")
    return "\n".join(lignes)


def matrice(r: dict, classes, libelles) -> str:
    c, n = r["confusion"], len(classes)
    abr = [libelles.get(x, x).split(" (")[0][:14] for x in classes]
    lignes = ["| | " + " | ".join(abr) + " |", "|---|" + "---|" * n]
    for i in range(n):
        if c[i].sum():
            lignes.append(f"| **{abr[i]}** | " + " | ".join(str(v) if v else "·" for v in c[i]) + " |")
    return "\n".join(lignes)


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--modele", default="tomate_mobilenetv3.onnx")
    ap.add_argument("--ancien", help="modèle précédent, pour comparaison")
    ap.add_argument("--classes", default="classes.json")
    ap.add_argument("--libelles", default="classes_fr.json")
    a = ap.parse_args()
    classes = json.loads(Path(a.classes).read_text(encoding="utf-8"))
    libelles = json.loads(Path(a.libelles).read_text(encoding="utf-8"))
    jeux = {nom: Path(d) for nom, d in JEUX.items() if Path(d).is_dir()}

    resultats = {nom: evaluer(a.modele, d, classes) for nom, d in jeux.items()}
    anciens = {nom: evaluer(a.ancien, d, classes) for nom, d in jeux.items()} if a.ancien else {}

    print(f"# Évaluation du CNN de diagnostic\n\nModèle `{Path(a.modele).name}` (ONNX Runtime, CPU), "
          "évalué avec le prétraitement du service de diagnostic, sur des images jamais vues à l'entraînement.\n")
    entete = "| Jeu d'images | Images | Exactitude | Réponses « fiables » (≥ 0,7) | Exactitude des réponses fiables |"
    print("## Résultats\n\n" + entete + "\n|---|---|---|---|---|")
    for nom, r in resultats.items():
        s = resume(r)
        ligne = f"| {nom} | {s['images']} | **{s['exactitude']:.1%}** | {s['fiables']:.0%} | {s['exactitude_fiables']:.1%} |"
        print(ligne)
    if anciens:
        print("\n## Comparaison avec le modèle précédent\n\n| Jeu d'images | Ancien modèle | Nouveau modèle |\n|---|---|---|")
        for nom in resultats:
            print(f"| {nom} | {resume(anciens[nom])['exactitude']:.1%} | **{resume(resultats[nom])['exactitude']:.1%}** |")
    toutes = [d for r in resultats.values() for d in r["durees"]]
    print(f"\nTemps d'inférence médian : {np.median(toutes) * 1000:.0f} ms par image (hors décodage).")
    for nom, r in resultats.items():
        print(f"\n## {nom}\n\n### Par classe\n\n{tableau_classes(r, classes, libelles)}")
        print(f"\n### Matrice de confusion (lignes : vraie classe, colonnes : prédiction)\n\n{matrice(r, classes, libelles)}")
    print("\n## Limites\n\n"
          "- PlantVillage : photos de laboratoire (feuille isolée, fond uni) ; ses chiffres surestiment les "
          "performances réelles.\n"
          "- PlantDoc : photos réelles mais peu nombreuses, collectées sur internet ; quelques étiquettes sont "
          "discutables et la classe « tache cible » n'y figure pas.\n"
          "- D'où le seuil de confiance de 0,7 et la confirmation du diagnostic par l'exploitant.")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/jk/ia/cnn/classes_fr.json
{
  "Tomato___Bacterial_spot": "Gale bactérienne (tache bactérienne)",
  "Tomato___Early_blight": "Alternariose (brûlure précoce)",
  "Tomato___Late_blight": "Mildiou",
  "Tomato___Leaf_Mold": "Cladosporiose (moisissure olive)",
  "Tomato___Septoria_leaf_spot": "Septoriose",
  "Tomato___Spider_mites Two-spotted_spider_mite": "Acariens (tétranyque tisserand)",
  "Tomato___Target_Spot": "Corynesporiose (tache cible)",
  "Tomato___Tomato_Yellow_Leaf_Curl_Virus": "Virus des feuilles jaunes en cuillère (TYLCV)",
  "Tomato___Tomato_mosaic_virus": "Virus de la mosaïque de la tomate (ToMV)",
  "Tomato___healthy": "Feuille saine"
}

In [ ]:
%%writefile /content/jk/service-ia/ia_service/images.py
"""Validation et prétraitement des photos de feuilles.

Le prétraitement reproduit celui de la validation à l'entraînement (ia/cnn/entrainement.py) :
redimensionnement du petit côté à 256, recadrage central 224, normalisation ImageNet, format NCHW.
"""
import io

import numpy as np
from PIL import Image, ImageOps, UnidentifiedImageError

TAILLE, REDIM = 224, 256
MOY = np.array([0.485, 0.456, 0.406], dtype=np.float32)
ECART = np.array([0.229, 0.224, 0.225], dtype=np.float32)
TAILLE_MAX_OCTETS = 5 * 1024 * 1024
COTE_MIN = 64
Image.MAX_IMAGE_PIXELS = 40_000_000      # protège contre les images compressées géantes


class ImageInvalide(ValueError):
    """Image refusée ; `status` est le code HTTP à renvoyer."""

    def __init__(self, status: int, detail: str):
        super().__init__(detail)
        self.status, self.detail = status, detail


def charger(donnees: bytes) -> Image.Image:
    if len(donnees) > TAILLE_MAX_OCTETS:
        raise ImageInvalide(413, "Image trop volumineuse (5 Mo maximum)")
    if not (donnees.startswith(b"\xff\xd8\xff") or donnees.startswith(b"\x89PNG\r\n\x1a\n")):
        raise ImageInvalide(415, "Format non pris en charge : JPEG ou PNG uniquement")
    try:
        with Image.open(io.BytesIO(donnees)) as verification:
            verification.verify()
        img = Image.open(io.BytesIO(donnees))
        img = ImageOps.exif_transpose(img)       # orientation des photos de téléphone
        img = img.convert("RGB")
    except (UnidentifiedImageError, OSError, SyntaxError, Image.DecompressionBombError) as e:
        raise ImageInvalide(422, "Image illisible ou corrompue") from e
    if min(img.size) < COTE_MIN:
        raise ImageInvalide(422, f"Image trop petite (minimum {COTE_MIN} px de côté)")
    return img


def pretraiter(img: Image.Image) -> np.ndarray:
    largeur, hauteur = img.size
    if largeur <= hauteur:
        nl, nh = REDIM, int(REDIM * hauteur / largeur)
    else:
        nl, nh = int(REDIM * largeur / hauteur), REDIM
    img = img.resize((nl, nh), Image.BILINEAR)
    gauche = int(round((nl - TAILLE) / 2.0))
    haut = int(round((nh - TAILLE) / 2.0))
    img = img.crop((gauche, haut, gauche + TAILLE, haut + TAILLE))
    x = (np.asarray(img, dtype=np.float32) / 255.0 - MOY) / ECART
    return x.transpose(2, 0, 1)[np.newaxis].astype(np.float32)

In [ ]:
%%writefile /content/jk/service-ia/ia_service/classifieur.py
"""Classification des maladies de la tomate avec le modèle ONNX entraîné."""
import hashlib
import json
import logging
import threading
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import onnxruntime as ort

log = logging.getLogger(__name__)
FICHIER_MODELE, FICHIER_CLASSES, FICHIER_LIBELLES = "tomate_mobilenetv3.onnx", "classes.json", "classes_fr.json"


class ModeleIndisponible(RuntimeError):
    pass


def softmax(logits: np.ndarray) -> np.ndarray:
    e = np.exp(logits - logits.max())
    return e / e.sum()


class Classifieur:
    """Charge le modèle une seule fois ; le recharge si le fichier ONNX est remplacé."""

    def __init__(self, dossier: Path, fils: int = 2):
        self.dossier, self.fils = Path(dossier), fils
        self.session: ort.InferenceSession | None = None
        self.classes: list[str] = []
        self.libelles: dict[str, str] = {}
        self.version: str | None = None
        self._signature = None
        self._verrou = threading.Lock()

    def _charger_si_besoin(self) -> None:
        """Charge le modèle au premier appel, ou si le fichier a changé depuis le dernier chargement."""
        chemin = self.dossier / FICHIER_MODELE
        if not chemin.exists() or not (self.dossier / FICHIER_CLASSES).exists():
            raise ModeleIndisponible("Modèle de diagnostic absent")
        stat = chemin.stat()
        signature = (stat.st_mtime_ns, stat.st_size)
        if signature == self._signature:
            return
        with self._verrou:
            if signature == self._signature:
                return
            options = ort.SessionOptions()
            options.intra_op_num_threads = self.fils
            session = ort.InferenceSession(str(chemin), options, providers=["CPUExecutionProvider"])
            classes = json.loads((self.dossier / FICHIER_CLASSES).read_text(encoding="utf-8"))
            if session.get_outputs()[0].shape[-1] != len(classes):
                raise ModeleIndisponible("Le nombre de classes ne correspond pas aux sorties du modèle")
            libelles = self.dossier / FICHIER_LIBELLES
            self.libelles = json.loads(libelles.read_text(encoding="utf-8")) if libelles.exists() else {}
            # L'empreinte couvre aussi le fichier de poids séparé produit par l'export ONNX.
            empreinte = hashlib.sha256(chemin.read_bytes())
            poids = chemin.with_name(chemin.name + ".data")
            if poids.exists():
                empreinte.update(poids.read_bytes())
            date = datetime.fromtimestamp(stat.st_mtime, timezone.utc).strftime("%Y-%m-%d")
            self.session, self.classes, self._signature = session, classes, signature
            # Architecture inscrite dans les métadonnées du fichier ONNX par entrainement.py (v2) ;
            # le premier modèle n'en avait pas : c'était un MobileNetV3-Small.
            architecture = session.get_modelmeta().custom_metadata_map.get("architecture", "mobilenetv3-small")
            self.version = f"{architecture}-{date}-{empreinte.hexdigest()[:8]}"
            log.info("Modèle chargé : %s", self.version)

    @property
    def pret(self) -> bool:
        try:
            self._charger_si_besoin()
            return True
        except ModeleIndisponible:
            return False

    def predire(self, x: np.ndarray, k: int = 3) -> dict:
        self._charger_si_besoin()
        entree = self.session.get_inputs()[0].name
        probabilites = softmax(self.session.run(None, {entree: x})[0][0].astype(np.float64))
        ordre = np.argsort(-probabilites)[:k]
        top = [{"classe": self.classes[i], "libelle": self.libelles.get(self.classes[i], self.classes[i]),
                "score": round(float(probabilites[i]), 4)} for i in ordre]
        return {**top[0], "alternatives": top[1:], "modele": self.version}

In [ ]:
!touch /content/jk/service-ia/ia_service/__init__.py

## Téléchargement des données (5 à 10 minutes)

In [ ]:
%cd /content/externe
# PlantVillage : seulement les dossiers tomate de la version « color »
!git clone -q --filter=blob:none --no-checkout https://github.com/spMohanty/PlantVillage-Dataset.git PlantVillage
!cd PlantVillage && git sparse-checkout init --no-cone && git sparse-checkout set 'raw/color/Tomato*' && git checkout -q master
!ls PlantVillage/raw/color | head -12

In [ ]:
%cd /content/externe
!git clone -q --depth 1 https://github.com/pratikkayal/PlantDoc-Dataset.git PlantDoc
!ls PlantDoc/train | grep -i tomato

In [ ]:
%cd /content/externe
# Pakistan : on n'extrait que les originaux (« Dataset (raw) »), pas les copies retouchées
!mkdir -p Mendeley_Pakistan && curl -sL -o Mendeley_Pakistan/pakistan.zip "https://data.mendeley.com/public-files/datasets/3mbnb82mxd/files/4844fda3-b66b-4b2a-bd83-775c0da974ea/file_downloaded"
!cd Mendeley_Pakistan && unzip -q -o pakistan.zip "Tomato Dataset/Dataset (raw)/*" && ls "Tomato Dataset/Dataset (raw)"

In [ ]:
# Bangladesh : le jeu ne publie pas les noms des dossiers ; chacun est reconnu par son nombre de photos,
# donné dans la description officielle (les dix effectifs sont tous différents)
import json, urllib.request
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

ouvrir = lambda url: urllib.request.urlopen(urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"}), timeout=60)
PAR_EFFECTIF = {394: "Tomato_Leaf_Curl_Virus", 307: "Spider_Mites", 66: "Leaf_Mold", 166: "Late_Blight",
                103: "Healthy", 204: "Early_Blight", 376: "Bacterial_Spot"}   # mineuse, insectes, cercosporiose écartées
d = json.load(ouvrir("https://data.mendeley.com/public-api/datasets/93h9p62kg4"))
par_dossier = defaultdict(list)
for f in d["files"]:
    par_dossier[f["folder_id"]].append(f)
a_faire = [(f, Path("/content/externe/Mendeley_Bangladesh") / PAR_EFFECTIF[len(fs)])
           for fs in par_dossier.values() if len(fs) in PAR_EFFECTIF for f in fs]

def telecharger(t):
    f, dossier = t
    dossier.mkdir(parents=True, exist_ok=True)
    for _ in range(3):
        try:
            (dossier / f["filename"]).write_bytes(ouvrir(f["content_details"]["download_url"]).read())
            return 1
        except Exception:
            pass
    return 0

with ThreadPoolExecutor(16) as ex:
    print(sum(ex.map(telecharger, a_faire)), "photos sur", len(a_faire))

## Préparation (mêmes graines que sur le PC : mêmes découpages entraînement / test)

In [ ]:
%cd /content/jk/ia/cnn
!python preparer_donnees.py --source /content/externe/PlantVillage/raw/color --sortie donnees | tail -3
!python preparer_plantdoc.py --source /content/externe/PlantDoc --sortie donnees_terrain
!python preparer_mendeley.py --pakistan "/content/externe/Mendeley_Pakistan/Tomato Dataset/Dataset (raw)" --bangladesh /content/externe/Mendeley_Bangladesh --sortie donnees_mendeley
# Sur le PC : PlantDoc 271 photos de test, Mendeley 428 photos de test
!find donnees_terrain/test -type f | wc -l; find donnees_mendeley/test -type f | wc -l

## Entraînement sur GPU

In [ ]:
%cd /content/jk/ia/cnn
!python entrainement.py --epoques 10 --images-par-epoque 8000 --processus 2 --sortie candidat/tomate_mobilenetv3.onnx

## Évaluation avec le prétraitement du service, puis téléchargement

In [ ]:
%cd /content/jk/ia/cnn
!PYTHONIOENCODING=utf-8 python evaluer.py --modele candidat/tomate_mobilenetv3.onnx --classes candidat/classes.json > candidat/evaluation.md
!sed -n 5,12p candidat/evaluation.md
!cd candidat && zip -q ../modele_cnn.zip tomate_mobilenetv3.onnx tomate_mobilenetv3.onnx.data classes.json evaluation.md
from google.colab import files
files.download("/content/jk/ia/cnn/modele_cnn.zip")